# Antisense NOT gate

Manual test harness for `engine.gates.antisense.AntisenseNotGate`. This family is
**implemented** (`available = True`), real ViennaRNA folding throughout.

The gate is generic over the payload gene: you construct it with `payload=<a CDS>`,
so the same code produces GFP-specific designs, mCherry-specific designs, or designs
for any other gene you pass in — nothing about the gene is hardcoded. This notebook
builds it against a real mCherry trigger window and a real GFP payload as a worked
example; swap either for whatever you're actually testing.

## The mechanism

An antisense RNA complementary to the payload's RBS and start region pairs with it
and blocks translation. Expression is **ON by default** and switched **OFF** when
the trigger appears — the inverse of a toehold, and what makes `NOT` buildable so a
circuit can use a down-regulated gene as an input.

Note the safety inversion: a *failed* antisense gate expresses the payload rather
than staying dark. And `predicted_leakage` here means residual expression when the
trigger **is** present — same metric name, opposite biological event.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

In [ ]:
# Standard EGFP CDS (DNA) — a real, well-known payload gene. `payload` accepts DNA
# or RNA; the gate converts it internally. Swap this for whatever gene you're testing.
GFP_CDS = (
    "ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCCTGGTCGAGCTGGACGGCGACGTAAACGGCCAC"
    "AAGTTCAGCGTGTCCGGCGAGGGCGAGGGCGATGCCACCTACGGCAAGCTGACCCTGAAGTTCATCTGCACCACCGGC"
    "AAGCTGCCCGTGCCCTGGCCCACCCTCGTGACCACCCTGACCTACGGCGTGCAGTGCTTCAGCCGCTACCCCGACCAC"
    "ATGAAGCAGCACGACTTCTTCAAGTCCGCCATGCCCGAAGGCTACGTCCAGGAGCGCACCATCTTCTTCAAGGACGAC"
    "GGCAACTACAAGACCCGCGCCGAGGTGAAGTTCGAGGGCGACACCCTGGTGAACCGCATCGAGCTGAAGGGCATCGAC"
    "TTCAAGGAGGACGGCAACATCCTGGGGCACAAGCTGGAGTACAACTACAACAGCCACAACGTCTATATCATGGCCGAC"
    "AAGCAGAAGAACGGCATCAAGGTGAACTTCAAGATCCGCCACAACATCGAGGACGGCAGCGTGCAGCTCGCCGACCAC"
    "TACCAGCAGAACACCCCCATCGGCGACGGCCCCGTGCTGCTGCCCGACAACCACTACCTGAGCACCCAGTCCGCCCTG"
    "AGCAAAGACCCCAACGAGAAGCGCGATCACATGGTCCTGCTGGAGTTCGTGACCGCCGCCGGGATCACTCTCGGCATG"
    "GACGAGCTGTACAAGTAA"
)

host = fx.Host.ECOLI          # ECOLI | YEAST | HUMAN
gate = fx.antisense(host=host, payload=GFP_CDS, real_fold=True)
fx.describe_gate(gate)        # available = True

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). The trigger below is a real, validated window — the most AU-rich 30 nt
stretch of the mCherry CDS, with no anti-SD motif, from `choose_trigger.py`'s scan.
Swap it via keyword for whatever trigger you're actually testing.

In [ ]:
# A NOT gates on a transcript that must be ABSENT, so the trigger is a repressor.
# This is the real, validated mCherry trigger window (nt 589-639 of the CDS).
mcherry_trigger = fx.sample_trigger(
    trigger_id="trig-mcherry-589",
    gene_id="mCherry",
    symbol="mCherry",
    sequence="UGAUGAACUUCGAGGACGGCGGCGUGGUGA",
    start_index=588,
)
triggers = fx.TriggerSet(activators=(), repressors=(mcherry_trigger,))
constraints = fx.sample_constraints()

for t in triggers.repressors:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — implemented

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()`

Cheap checks only, no folding: one input, and it must be a repressor; the trigger
must be long enough to supply a UTR arm + RBS/Kozak + spacer; and accessible enough
(read from stage 2's numbers) for an antisense arm to plausibly pair.

In [ ]:
fx.attempt('is_compatible', lambda: gate.is_compatible(triggers, constraints))

## `generate_designs()`

Sweeps UTR length x spacer length x window offset along the trigger. No folding here
— cheap, structural construction only. Each yielded design already carries the real
`GFP_CDS` payload head spliced in after AUG.

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()`

This is where real ViennaRNA folding happens: partition-function accessibility of
the switch alone (ON) and of the switch+trigger complex (OFF/leakage), plus the
hybridisation energy between switch and trigger.

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

Called on a real generated design now that `generate_designs()` produces one,
rather than `fx.sample_design(...)`'s hand-built stand-in.

In [ ]:
design = designs[0] if designs else fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('architecture:', design.architecture)
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Folding engine

This notebook builds the gate with `real_fold=True`, so everything above already
used real ViennaRNA — the `evaluate_design()` numbers are genuine partition-function
predictions, not placeholders. For a faster (fake) run while just poking at
construction logic, drop `real_fold=True`:

```python
gate = fx.antisense(host=host, payload=GFP_CDS)   # fx.StubFoldEngine, no ViennaRNA
```